In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [ ]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [ ]:
results

In [ ]:
from vector_search.rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os

LLM_MODEL = "MiniMax-M2.7"

load_dotenv(override=True)

api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_BASE_URL")
if not api_key or not base_url:
    raise SystemExit(
        "Missing env vars. Set OPENAI_API_KEY and OPENAI_BASE_URL in .env."
    )

openai_client = OpenAI(api_key=api_key, base_url=base_url)

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
    model=LLM_MODEL
)

In [ ]:
vector_assistant.rag('the program has already begun, can I still sign up?')


In [ ]:
vector_assistant.rag('whats queen gambit?')
